## Tool Calling Basics

In [29]:
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

True

In [30]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI()

In [31]:
llm.invoke("How will the weather be in Delhi today?")

AIMessage(content="I'm sorry, but I am not able to provide real-time weather updates. I recommend checking a reliable weather forecasting website or app for the most up-to-date information on the weather in Delhi today.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 40, 'prompt_tokens': 16, 'total_tokens': 56, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DVEdE6yFhOfuGzx15gn9y2puV8NpJ', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--c7339cb3-5b2b-4e34-a19f-4b631da577c1-0', usage_metadata={'input_tokens': 16, 'output_tokens': 40, 'total_tokens': 56, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'r

In [32]:
from langchain_core.tools import tool

@tool
def get_weather(location: str):
    """Call to get the current weather."""
    if location.lower() in ["delhi", "new delhi"]:
        return "The weather in Delhi is sunny with a high of 30°C."
    else:
        return f"Sorry, I don't have weather information for {location}."
    
@tool
def check_seating_availability(location: str, seating_type: str):
    """Call to check seating availability."""
    if location.lower() in ["delhi", "new delhi"] and seating_type.lower() == "outdoor":
        return "Yes, we still have seats available outdoors."
    elif location.lower() in ["delhi", "new delhi"] and seating_type.lower() == "indoor":
        return "Yes, we have seats available indoors."
    else:
        return f"Sorry, I don't have information about {seating_type} seating for {location}."
    
tools = [get_weather, check_seating_availability]

In [33]:
llm_with_tools = llm.bind_tools(tools)

In [34]:
result = llm_with_tools.invoke("How will the weather be in Delhi today?")
result

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 15, 'prompt_tokens': 83, 'total_tokens': 98, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DVEdOaR0YNc1Dmm6KdybPSiP9cEqG', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--c8f8b85a-f0a4-4470-b2a8-99828083ed85-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Delhi'}, 'id': 'call_W7NmUbAF6vSwrnUdlN5dDk3o', 'type': 'tool_call'}], usage_metadata={'input_tokens': 83, 'output_tokens': 15, 'total_tokens': 98, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [35]:
result.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Delhi'},
  'id': 'call_W7NmUbAF6vSwrnUdlN5dDk3o',
  'type': 'tool_call'}]

In [36]:
result = llm_with_tools.invoke("What's the weather like in Delhi and do you have outdoor seating available?")

In [37]:
result

AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 89, 'total_tokens': 144, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DVEddsiqj4iU8UYMTCRLBm4nK63pJ', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--b5f0dd85-8ac8-46de-9b0b-619a33c48e80-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'Delhi'}, 'id': 'call_ngBL9yAQNm5yxwMwDiMAG3Bo', 'type': 'tool_call'}, {'name': 'check_seating_availability', 'args': {'location': 'Delhi', 'seating_type': 'outdoor'}, 'id': 'call_6cIJxOq6zbswz8i0eGnXNGyh', 'type': 'tool_call'}], usage_metadata={'input_tokens': 89, 'output_tokens': 55, 'total_tokens': 144

In [38]:
result.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'Delhi'},
  'id': 'call_ngBL9yAQNm5yxwMwDiMAG3Bo',
  'type': 'tool_call'},
 {'name': 'check_seating_availability',
  'args': {'location': 'Delhi', 'seating_type': 'outdoor'},
  'id': 'call_6cIJxOq6zbswz8i0eGnXNGyh',
  'type': 'tool_call'}]

In [46]:
from langchain_core.messages import HumanMessage, ToolMessage

messages = [
    HumanMessage(
        "How will the weather be in delhi today? Do you still have outdoor seating available?"
    )
]

llm_output = llm_with_tools.invoke(messages)
messages.append(llm_output)

In [47]:
messages

[HumanMessage(content='How will the weather be in delhi today? Do you still have outdoor seating available?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 92, 'total_tokens': 147, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DVEiqCVnpI4yDdJ2fYsuBGQ93RFJE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--4fbc9bcc-ce33-461e-8102-4800ac1265b9-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'delhi'}, 'id': 'call_wX2w2ajr863489udnWfpPkbm', 'type': 'tool_call'}, {'name': 'check_seating_availability', 'args': {'location': 'delhi', 'seating_t

In [48]:
tool_mapping = {
    "get_weather": get_weather,
    "check_seating_availability": check_seating_availability
}

In [49]:
llm_output.tool_calls

[{'name': 'get_weather',
  'args': {'location': 'delhi'},
  'id': 'call_wX2w2ajr863489udnWfpPkbm',
  'type': 'tool_call'},
 {'name': 'check_seating_availability',
  'args': {'location': 'delhi', 'seating_type': 'outdoor'},
  'id': 'call_FRILoE15PaCLzL9AeAajoGBO',
  'type': 'tool_call'}]

In [50]:
for tool_call in llm_output.tool_calls:
    tool = tool_mapping[tool_call["name"].lower()]
    tool_output = tool.invoke(tool_call["args"])
    messages.append(ToolMessage(tool_output, tool_call_id=tool_call["id"]))


In [51]:
messages

[HumanMessage(content='How will the weather be in delhi today? Do you still have outdoor seating available?', additional_kwargs={}, response_metadata={}),
 AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 55, 'prompt_tokens': 92, 'total_tokens': 147, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DVEiqCVnpI4yDdJ2fYsuBGQ93RFJE', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--4fbc9bcc-ce33-461e-8102-4800ac1265b9-0', tool_calls=[{'name': 'get_weather', 'args': {'location': 'delhi'}, 'id': 'call_wX2w2ajr863489udnWfpPkbm', 'type': 'tool_call'}, {'name': 'check_seating_availability', 'args': {'location': 'delhi', 'seating_t

In [52]:
llm_with_tools.invoke(messages)

AIMessage(content='The weather in Delhi today is sunny with a high of 30°C. And yes, outdoor seating is still available.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 182, 'total_tokens': 207, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DVEkL4CXDRiYvB3SIMcnHxtCwwskY', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--373cb0f8-d688-4511-8d67-21330e7ceade-0', usage_metadata={'input_tokens': 182, 'output_tokens': 25, 'total_tokens': 207, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})